# 01 · Preparación de datos (VisDrone-DET, copia de Kaggle)
Objetivo: convertir YOLO → COCO JSON, crear subconjuntos y medir la densidad de objetos.
**Decisiones a registrar en `DESIGN_LOG.md`:** subconjunto usado (D4), orden de clases de la copia (D14), tratamiento de cajas inválidas (D8).

Los JSON pequeños (`train_sub.json`, `val.json`, `val_sub.json`) se **commitean** al repo para que los notebooks 02 y 03 corran en otra sesión sin repetir la conversión.

In [ ]:
# Configuración común (Colab/Kaggle o local). Ajusta REPO_URL una sola vez.
import os, sys
REPO_URL = 'https://github.com/<usuario>/detr-query-saturation.git'
if not os.path.exists('../src/common.py'):            # no estamos dentro de notebooks/ -> clonar
    if not os.path.exists('detr-query-saturation/src'):
        !git clone -q {REPO_URL}
    os.chdir('detr-query-saturation/notebooks')
    !pip -q install -r ../requirements.txt
sys.path.insert(0, '../src')
from common import *
import torch, json, pandas as pd, matplotlib.pyplot as plt
os.makedirs('../results', exist_ok=True); os.makedirs('../data', exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; SEED = 42; set_seed(SEED)
print(DEVICE); print(env_report('../results/entorno.txt'))   # versiones exactas -> README

In [ ]:
ROOT = '/kaggle/input/<dataset-visdrone>'
for r, d, f in os.walk(ROOT):
    if r[len(ROOT):].count(os.sep) <= 3: print(r, len(f), 'archivos')

In [ ]:
# Rutas de la copia de Kaggle de VisDrone (ajusta tras revisar el árbol en 01)
TRAIN_IMG, TRAIN_LBL = f'{ROOT}/<train/images>', f'{ROOT}/<train/labels>'
VAL_IMG,   VAL_LBL   = f'{ROOT}/<val/images>',   f'{ROOT}/<val/labels>'

# D14 · Verifica el orden de clases de la copia: debe coincidir con VISDRONE_CLASSES,
# porque COCO_TO_VISDRONE (evaluación zero-shot) depende de ese orden.
import glob
ymls = glob.glob(f'{ROOT}/**/*.yaml', recursive=True); print(ymls)
if ymls:
    names = read_yolo_names(ymls[0]); print(names)
    if [n.lower() for n in names] != VISDRONE_CLASSES:
        print('⚠️ ORDEN DISTINTO: usa class_names=names en yolo_to_coco y corrige COCO_TO_VISDRONE en src/common.py')

In [ ]:
print(yolo_to_coco(TRAIN_IMG, TRAIN_LBL, '../data/train.json'))
print(yolo_to_coco(VAL_IMG,   VAL_LBL,   '../data/val.json'))

In [ ]:
# D4 · Subconjunto de entrenamiento (cómputo limitado). Validación: las 548 imágenes completas
# (la inferencia es barata y más imágenes densas dan un análisis por grupos más estable).
N_TRAIN, N_VAL = 600, None          # None = todas
print(make_subset('../data/train.json', N_TRAIN, '../data/train_sub.json', SEED))
print(make_subset('../data/val.json',   N_VAL,   '../data/val_sub.json',   SEED))

In [ ]:
# Distribución por clase y cobertura del mapeo COCO -> VisDrone (zero-shot)
d = json.load(open('../data/val_sub.json'))
C = pd.Series([a['category_id'] for a in d['annotations']]).value_counts().sort_index()
C.index = [VISDRONE_CLASSES[i - 1] for i in C.index]
cub = sum(C[VISDRONE_CLASSES[v - 1]] for v in set(COCO_TO_VISDRONE.values()) if VISDRONE_CLASSES[v - 1] in C)
print(C.to_string()); print(f'cajas cubiertas por clases con equivalente COCO: {cub / C.sum():.1%}')
C.to_csv('../results/cajas_por_clase_val.csv')

In [ ]:
# Densidad: objetos por imagen y techo de recall (sin modelo) para N = 100, 300, 500
cnt = objects_per_image('../data/val_sub.json'); n = pd.Series(cnt)
print(n.describe())
rows = []
for N in (100, 300, 500):
    rows.append({'N': N, '% imágenes con n > N': round(100 * (n > N).mean(), 1),
                 'techo de recall (dataset)': round(100 * oracle_recall(cnt, N), 1)})
    print(f'N={N}: imágenes con más de N objetos = {(n > N).mean():.1%} | techo de recall = {oracle_recall(cnt, N):.1%}')
pd.DataFrame(rows).to_csv('../results/techo_recall.csv', index=False)
groups, (q1, q2) = density_groups('../data/val_sub.json')
print({k: len(v) for k, v in groups.items()}, 'cortes de terciles:', q1, q2)
n.hist(bins=50); plt.axvline(100, color='k', ls='--', label='N = 100'); plt.legend()
plt.xlabel('objetos por imagen'); plt.ylabel('imágenes'); plt.savefig('../results/hist_objetos.png', dpi=150, bbox_inches='tight')

**Lectura:** si el % de imágenes con más de 100 objetos es muy bajo, H1 pierde fuerza. Anótalo en el log, sea cual sea el resultado.
El techo es una cota analítica (detector perfecto con N queries), no una medición.